# Semantic Document Search — Pipeline Demo

End-to-end walkthrough of the production stack:

- **FastAPI** service exposing `/ingest` (FSM-driven) and `/search` (functional pipeline)
- **Qdrant Cloud** with named vectors (dense `bge-m3` + sparse `Qdrant/bm25`)
- **OpenRouter** as the unified LLM gateway (chat, embeddings, rerank)
- **Three retrieval strategies** swappable per request: `dense_only`, `hybrid` (RRF fusion), `hybrid_rerank` (Cohere `rerank-3.5` over fused candidates)

What this notebook shows, in order:

1. **Health check** — confirms the service and Qdrant connectivity
2. **Ingestion** — POST a fresh dev.to URL, poll until `completed`, render the FSM transition history
3. **Retrieval side-by-side** — same query against all three strategies, inspect rank order and scores
4. **Evaluation summary** — IR metrics from the 32-query golden set (`docs/eval-results.txt`)

**Pre-requisites (run in another terminal):**

```bash
make migrate   # one-time SQLite schema
make dev       # uvicorn on http://127.0.0.1:8000
```

The seed corpus is already loaded in Qdrant (50 dev.to articles via `make seed`).

## 1. Setup & Health Check

In [1]:
from __future__ import annotations

import time
from pathlib import Path

import httpx
import pandas as pd
from IPython.display import Markdown, display

APP_URL = "http://127.0.0.1:8000"

# Fail fast if the service isn't up. /health surfaces version, commit
# SHA, uptime, and a DB liveness probe — useful for sanity in a demo.
with httpx.Client(timeout=5.0) as client:
    r = client.get(f"{APP_URL}/health")
    r.raise_for_status()
    health = r.json()
health

{'status': 'ok',
 'version': '0.2.0',
 'commit': 'dev',
 'uptime_seconds': 9.21,
 'environment': 'development',
 'db': 'ok'}

## 2. Ingestion — Live FSM Walkthrough

The ingestion job is driven by the [`transitions`](https://github.com/pytransitions/transitions) FSM library. The state machine flows:

```
pending → fetching → parsing → chunking → embedding → indexing → completed
                                                                ↘ failed (on error)
```

(API returns these names verbatim — lowercase string enum.)

Each transition is persisted to SQLite (`job_transitions` table) so the API can return the full timeline at any point — even after the job is terminal. **Re-runs are idempotent**: the `job_id` is `sha256(url)[:16]`, so a second POST of the same URL detects the existing terminal job and returns it instead of spawning a duplicate pipeline. That's the behavior the next two cells demonstrate (the history dataframe still shows every step, even though the job was indexed on a previous run).

In [2]:
# Demo URL: a dev.to post NOT in the seed corpus, so we can show the
# pipeline working from cold. (If you re-run, the deterministic job_id
# means it'll detect the existing job and short-circuit — also a feature.)
DEMO_URL = "https://dev.to/the_nortern_dev/the-emotional-terror-of-closing-a-browser-tab-122k"

with httpx.Client(timeout=30.0) as client:
    r = client.post(f"{APP_URL}/ingest", json={"source_url": DEMO_URL})
    r.raise_for_status()
    job = r.json()
    job_id = job["job_id"]
    print(f"job_id={job_id}")
    print(f"initial state={job['state']}")

    # Poll until terminal (completed or failed). Real clients would
    # webhook — this is just to render progression in the notebook.
    terminal = {"completed", "failed"}
    while True:
        r = client.get(f"{APP_URL}/ingest/jobs/{job_id}")
        r.raise_for_status()
        job = r.json()
        if job["state"] in terminal:
            break
        time.sleep(0.5)

    print(f"final state={job['state']}")
    if job.get("error"):
        print(f"error={job['error']}")

job_id=f0fc5c1e48a6a00a
initial state=completed
final state=completed


In [3]:
# Render the transition history as a timeline.
history = pd.DataFrame(job["history"])
history["at"] = pd.to_datetime(history["at"])
history["elapsed_ms"] = (
    ((history["at"] - history["at"].iloc[0]).dt.total_seconds() * 1000).round(0).astype(int)
)
history

,state,at,elapsed_ms
0,fetching,2026-04-18 22:06:39,0
1,parsing,2026-04-18 22:06:39,0
2,chunking,2026-04-18 22:06:39,0
3,embedding,2026-04-18 22:06:39,0
4,indexing,2026-04-18 22:06:42,3000
5,completed,2026-04-18 22:06:43,4000


## 3. Retrieval — Strategies Side by Side

Same query, three strategies, top-3 results each. The `score` column is the score from the *last* stage that touched the candidate, so scales aren't comparable across strategies — clients use scores to order *within* a response, not across.

- **`dense_only`** — `bge-m3` cosine over the dense vector
- **`hybrid`** — Qdrant Query API fuses dense + BM25 sparse via Reciprocal Rank Fusion (server-side)
- **`hybrid_rerank`** — Cohere `rerank-3.5` reorders the fused top-N over the *parent* chunk text (parent-child retrieval pattern)

In [4]:
STRATEGIES = ["dense_only", "hybrid", "hybrid_rerank"]


def search(client: httpx.Client, q: str, strategy: str, k: int = 3) -> list[dict]:
    r = client.get(
        f"{APP_URL}/search",
        params={"q": q, "strategy": strategy, "top_k": k},
        timeout=60.0,
    )
    r.raise_for_status()
    return r.json()["results"]


def render_query(q: str) -> None:
    display(Markdown(f"### Query: _{q!r}_"))
    rows = []
    with httpx.Client() as client:
        for s in STRATEGIES:
            for rank, hit in enumerate(search(client, q, s), start=1):
                rows.append(
                    {
                        "strategy": s,
                        "rank": rank,
                        "score": round(hit["score"], 3),
                        "title": hit["title"][:70],
                    }
                )
    df = pd.DataFrame(rows).pivot(index="rank", columns="strategy", values=["title", "score"])
    display(df)

In [5]:
# Keyword-heavy query — BM25 should match exact terms cleanly.
render_query("replace Redis with Postgres for caching")

### Query: _'replace Redis with Postgres for caching'_

title  \
strategy                                         dense_only   
rank                                                          
1         I Replaced Redis with PostgreSQL (And It's Fas...   
2         I Replaced Redis with PostgreSQL (And It's Fas...   
3         I Replaced Redis with PostgreSQL (And It's Fas...   

                                                             \
strategy                                             hybrid   
rank                                                          
1         I Replaced Redis with PostgreSQL (And It's Fas...   
2         I Replaced Redis with PostgreSQL (And It's Fas...   
3         I Replaced Redis with PostgreSQL (And It's Fas...   

                                                                 score         \
strategy                                      hybrid_rerank dense_only hybrid   
rank                                                                            
1         I Replaced Redis with PostgreSQL (And It's Fas...        0.5    1.0   
2         I Replaced Redis with PostgreSQL (And It's Fas...      0.333  0.667   
3         I Replaced Redis with PostgreSQL (And It's Fas...       0.25  0.375   

                        
strategy hybrid_rerank  
rank                    
1                0.897  
2                0.853  
3                 0.55

In [6]:
# Conceptual paraphrase — dense embeddings shine when terms don't match literally.
render_query("how AI agents remember context across sessions")

### Query: _'how AI agents remember context across sessions'_

title  \
strategy                                         dense_only   
rank                                                          
1         Adding Persistent Memory to Claude Code with c...   
2                  your agent can think. it can't remember.   
3                              🧠Maybe I Just Do Not Get It!   

                                                             \
strategy                                             hybrid   
rank                                                          
1         Adding Persistent Memory to Claude Code with c...   
2                              🧠Maybe I Just Do Not Get It!   
3                  your agent can think. it can't remember.   

                                                                 score         \
strategy                                      hybrid_rerank dense_only hybrid   
rank                                                                            
1         Adding Persistent Memory to Claude Code with c...        0.5  0.833   
2                  your agent can think. it can't remember.      0.333    0.5   
3                              🧠Maybe I Just Do Not Get It!       0.25  0.424   

                        
strategy hybrid_rerank  
rank                    
1                0.599  
2                0.468  
3                0.333

In [7]:
# Ambiguous query — useful to see where the reranker pulls ahead.
render_query("WebGPU browser 3D graphics capabilities")

### Query: _'WebGPU browser 3D graphics capabilities'_

title  \
strategy                                         dense_only   
rank                                                          
1         Most Apps Are Slower Than They Need to Be — He...   
2         Most Apps Are Slower Than They Need to Be — He...   
3         Why WebGPU Feels Like the Future of the Web (L...   

                                                             \
strategy                                             hybrid   
rank                                                          
1         Most Apps Are Slower Than They Need to Be — He...   
2         Why WebGPU Feels Like the Future of the Web (L...   
3         Most Apps Are Slower Than They Need to Be — He...   

                                                                 score         \
strategy                                      hybrid_rerank dense_only hybrid   
rank                                                                            
1         Why WebGPU Feels Like the Future of the Web (L...        0.5   0.75   
2         Most Apps Are Slower Than They Need to Be — He...      0.333   0.75   
3         Why WebGPU Feels Like the Future of the Web (L...       0.25  0.433   

                        
strategy hybrid_rerank  
rank                    
1                0.591  
2                0.576  
3                0.502

## 4. Evaluation — Golden Set Results

Source of truth: `app/evaluation/queries.yaml` (32 queries, mix of keyword-heavy / paraphrase / adversarial). Run via `make eval` against a live service. Results are checked into `docs/eval-results.txt`.

Headline finding: with stemming + stopword-aware BM25 (FastEmbed `Qdrant/bm25`), **`hybrid` matches `dense_only`** rather than dragging it down — and the reranker delivers a clean **+6.2pp on P@1** over both baselines, which is the rerank value claim from the literature.

In [8]:
# Resolve the eval results regardless of whether Jupyter was started
# from the repo root or from notebooks/.
candidates = [Path("docs/eval-results.txt"), Path("../docs/eval-results.txt")]
eval_path = next((p for p in candidates if p.exists()), None)
if eval_path is None:
    raise FileNotFoundError(
        f"docs/eval-results.txt not found. Tried: {[str(p) for p in candidates]}"
    )
lines = eval_path.read_text(encoding="utf-8").splitlines()

# The summary table is at the top — print just it (everything before
# the per-query breakdown).
for line in lines:
    if line.startswith("Per-query breakdown"):
        break
    print(line)

Running 32 queries x 3 strategies against http://127.0.0.1:8000

STRATEGY         P@1     R@1     R@3     R@5     R@10    MRR     NDCG@5   NDCG@10  N
------------------------------------------------------------------------------------
dense_only       0.750   0.646   0.839   0.901   0.911   0.827   0.826    0.831    32
hybrid           0.750   0.646   0.839   0.901   0.911   0.827   0.826    0.831    32
hybrid_rerank    0.812   0.724   0.839   0.911   0.911   0.858   0.856    0.856    32



### What this tells us

| Strategy | P@1 | R@5 | MRR | NDCG@5 |
|---|---|---|---|---|
| `dense_only` | 0.750 | 0.901 | 0.827 | 0.826 |
| `hybrid` (RRF) | 0.750 | 0.901 | 0.827 | 0.826 |
| `hybrid_rerank` | **0.812** | **0.911** | **0.858** | **0.856** |

- **Dense vs hybrid tie** is expected on a corpus this size with topical (not keyword-pinned) queries — RRF fusion finds the same documents but reorders rarely enough to matter at k=5/10. Both win where the other one wins.
- **Reranker pulls ahead on P@1 / MRR / NDCG** — that's the cross-encoder doing exactly what cross-encoders do: sharper relevance judgments at the top of the list, which is where users look.
- **R@10 = 0.911 is the recall ceiling** of the indexed corpus for these queries. The reranker can't surface a document that retrieval never returned, so further gains have to come from upstream (better chunking, query expansion, more diverse seed corpus).

See `app/evaluation/runner.py` for the harness and `app/evaluation/metrics.py` for metric definitions (binary relevance, document-level dedup before scoring).